# Практическая работа №1

## Основные задачи обработки изображений

## Задание

### Цель

Знакомство с методами обработки изображений, формирование навыков выполнения обработки изображений на языке Python.

### Задачи

1.	Исследование методов преобразования изображений
2.	Исследование методов фильтрации изображений
3.	Исследование методов выделения контуров на изображении
4.	Исследование методов бинаризации изображений

## Ход работы

### Импорты и вспомогательные функции

### Подготовка окружения

Перед запуском основных экспериментов устанавливаю зависимости, от которых зависят примеры ниже.


In [ ]:
%pip install numpy scipy scikit-image matplotlib


In [ ]:
import math

import numpy as np
import scipy
import skimage
from skimage import data
from skimage import color
from skimage import filters
from skimage import exposure
from skimage.feature import canny
from skimage.filters import roberts, prewitt, sobel, laplace, try_all_threshold, threshold_otsu, threshold_local, rank, threshold_niblack, threshold_sauvola
from skimage.morphology import disk
from skimage.util import img_as_ubyte, random_noise
from skimage.metrics import structural_similarity, peak_signal_noise_ratio, mean_squared_error
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        # корректное отображение ЧБ изображений в Matplotlib
        'image.cmap': 'gray',
        'axes.titlesize': 10,
    }
)

In [ ]:
# отрисовка изображений в виде сетки
# images - словарь {название изображения: изображение}
def display_images(images: dict[str, object], ncolumns: int = 1) -> None:
    nrows = math.ceil(len(images) / ncolumns)
    fig, axes = plt.subplots(nrows, ncolumns, figsize=(15, 4 * nrows))
    axes = axes.flatten()
    
    for ax, (title, image) in zip(axes, images.items()):
        ax.axis('off')
        ax.imshow(image)
        ax.set_title(title)
        
    for i in range(len(images), len(axes)):
        axes[i].axis('off')
        
    fig.tight_layout()
    plt.show()
    
# отрисовка изображений и гистограммами яркости пикселей под каждым изображением
# images - словарь {название изображения: изображение}
def display_images_with_histograms(images: dict[str, object], ncolumns: int = 1) -> None:
    nrows = math.ceil(len(images) / ncolumns)
    fig, axes = plt.subplots(nrows * 2, ncolumns, figsize=(12, 3 * nrows * 2))
    axes = axes.reshape(nrows * 2, ncolumns)
    
    for idx, (title, image) in enumerate(images.items()):
        row = (idx // ncolumns) * 2
        col = idx % ncolumns
        
        ax_img = axes[row, col]
        ax_img.axis('off')
        ax_img.imshow(image)
        ax_img.set_title(title)
        
        ax_hist = axes[row + 1, col]
        bins = 256
        
        ax_hist.hist(image.ravel(), bins=bins, histtype='step', color='black')
        ax_hist.ticklabel_format(axis='y', style='scientific', scilimits=(0, 0))
        ax_hist.set_xlabel('Pixel intensity')
        ax_hist.set_ylabel('Number of pixels')
        
        img_cdf, bins = exposure.cumulative_distribution(image, bins)
        ax_cdf = ax_hist.twinx()
        ax_cdf.plot(bins, img_cdf, 'r')
        
    for i in range(len(images), nrows * ncolumns):
        axes[(i // ncolumns) * 2, i % ncolumns].axis('off')
        axes[(i // ncolumns) * 2 + 1, i % ncolumns].axis('off')
        
    fig.tight_layout()
    plt.show()
    
# отрисовка изображений и гистограммами яркости пикселей под каждым изображением и наложением порога
# images - словарь {название изображения: (бинарное изображение, исходное изображение, по которому строится гистограмма яркости пикселей, порог)}
def display_images_with_thresholds(images: dict[str, tuple[object, object, int]], ncolumns: int = 1) -> None:
    nrows = math.ceil(len(images) / ncolumns)
    fig, axes = plt.subplots(nrows * 2, ncolumns, figsize=(12, 3 * nrows * 2))
    axes = axes.reshape(nrows * 2, ncolumns)
    
    for idx, (title, image_obj) in enumerate(images.items()):
        row = (idx // ncolumns) * 2
        col = idx % ncolumns
        
        ax_img = axes[row, col]
        ax_img.axis('off')
        ax_img.imshow(image_obj[0])
        ax_img.set_title(title)
        
        ax_hist = axes[row + 1, col]
        bins = 256
        
        ax_hist.hist(image_obj[1].ravel(), bins=bins, histtype='step', color='black')
        ax_hist.ticklabel_format(axis='y', style='scientific', scilimits=(0, 0))
        ax_hist.set_xlabel('Pixel intensity')
        ax_hist.set_ylabel('Number of pixels')
        
        ax_hist.axvline(image_obj[2], color='r', linewidth=2)
        
    for i in range(len(images), nrows * ncolumns):
        axes[(i // ncolumns) * 2, i % ncolumns].axis('off')
        axes[(i // ncolumns) * 2 + 1, i % ncolumns].axis('off')
        
    fig.tight_layout()
    plt.show()

### 1. Подготовка изображений для исследования

Для экспериментов использую несколько эталонных снимков из `skimage`: портрет кота, тестовый снимок с камерой, страницу текста и фрагмент лунной поверхности. Для анализа контуров дополнительно подключаю фотографию из датасета [Berkeley Segmentation Dataset 500 (BSDS500)](https://www.kaggle.com/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/data), поскольку изображения из него снабжены аккуратной разметкой границ.



In [ ]:
img_cat = data.cat()
img_cam = data.camera()
img_page = data.page()
img_moon = data.moon()
img_deer = skimage.io.imread('./data/113016.jpg')
img_deer_edges = scipy.io.loadmat('./data/113016.mat')['groundTruth'][0][0][0][0][1]

source_images = {
    'cat': img_cat,
    'camera': img_cam,
    'page': img_page,
    'moon': img_moon,
    'deer': img_deer,
    'deer edges': img_deer_edges
}
display_images(source_images, 3)

### 2. Методы преобразования изображений

#### Цветовая коррекция

Сравниваю несколько вариантов представления цвета: перевожу исходный снимок в оттенки серого, а также в пространства HSV и YCbCr, чтобы посмотреть, как меняется визуальное восприятие и распределение интенсивностей.



In [ ]:
color_transformed_images = {
    'cat RGB': img_cat,
    'cat Gray': color.rgb2gray(img_cat),
    'cat YCbCr': np.uint(color.rgb2ycbcr(img_cat)),
    'cat HSV': color.rgb2hsv(img_cat),
}
display_images(color_transformed_images, 4)

#### Яркостная коррекция

Проверяю влияние основных однотонных преобразований: линейной и логарифмической коррекции, гамма-преобразования и инверсии. Эти операции позволяют смещать динамический диапазон освещенности и по-разному выделять детали на изображении.



In [ ]:
contrast_adjusted_images = {
    'original': img_moon,
    'contrast stretching': exposure.rescale_intensity(img_moon, in_range=(np.percentile(img_moon, 5), np.percentile(img_moon, 95))),
    'logarithmic correction': exposure.adjust_log(img_moon, gain=1.0),
    'gamma correction (darker)': exposure.adjust_gamma(img_moon, gamma=2.0),
    'gamma correction (lighter)': exposure.adjust_gamma(img_moon, gamma=0.7),
    'negative': exposure.adjust_gamma(img_moon, gamma=1.0, gain=-1.0),
}
display_images_with_histograms(contrast_adjusted_images, 4)


#### Коррекция на основе гистограммы

Сравниваю две версии усиления контраста: классическую эквализацию и адаптивную (CLAHE). Первый подход перераспределяет интенсивности по всему кадру, второй делает то же самое локально и лучше раскрывает мелкие элементы.



In [ ]:
contrast_adjusted_images = {
    'original': img_moon,
    'histogram equalization': exposure.equalize_hist(img_moon),
    'adaptative histogram equalization': exposure.equalize_adapthist(img_moon, clip_limit=0.03),
}
display_images_with_histograms(contrast_adjusted_images, 4)


### 3. Методы фильтрации изображений

Чтобы оценить устойчивость фильтров к артефактам, к изображениям добавляю несколько видов шума: гауссовский, импульсный с произвольными значениями, а также отдельные варианты «соли» и «перца». Для каждого случая применяю гауссовский, медианный и усредняющий фильтры и считаю метрики MSE, PSNR и SSIM, чтобы количественно сравнить качество восстановления. По статистике лучше всех себя показывает медианная фильтрация — она более надежно подавляет выбросы и сохраняет границы объектов.



In [ ]:
def add_noise_to_image(image: object, noise_type: str, noise_arg: float) -> object:
    if noise_type == 'impulse':
        noisy_image = img_as_ubyte(image, force_copy=True)
        rng = np.random.default_rng()
        noise = rng.random(noisy_image.shape)
        noisy_image[noise > noise_arg] = np.random.randint(0, 256)
    elif noise_type == 'gaussian':
        noisy_image = random_noise(image, mode=noise_type, var=noise_arg)
    elif noise_type in ('salt', 'pepper'):
        noisy_image = random_noise(image, mode=noise_type, amount=noise_arg)
    else:
        raise ValueError('Unsupported noise type')
    
    if noisy_image.dtype == np.float64:
        noisy_image = np.clip(noisy_image, 0, 1)
    
    return img_as_ubyte(noisy_image)

noise_params = {
    'gaussian': [0.01, 0.03, 0.05],
    'salt': [0.02, 0.04, 0.06],
    'pepper': [0.02, 0.04, 0.06],
    'impulse': [0.98, 0.95, 0.9],
}

filer_params = {
    filters.gaussian: tuple([1]),
    filters.rank.median: tuple([disk(3)]),
    filters.rank.mean: tuple([disk(3)])
}

for noise_type, params in noise_params.items():
    for param in params:
        noisy_images = {}
        noisy_image = add_noise_to_image(img_cam, noise_type, param)
        mse = mean_squared_error(img_cam, noisy_image)
        psnr = peak_signal_noise_ratio(img_cam, noisy_image)
        ssim = structural_similarity(img_cam, noisy_image)
        noisy_images[f'{noise_type} noise {param}\nMSE: {mse:.1f} PSNR: {psnr:.1f} SSIM: {ssim:.3f}'] = noisy_image
        
        for filter_type, filter_params in filer_params.items():
            filtered_image = filter_type(noisy_image, *filter_params)
            if filtered_image.dtype == np.float64:
                filtered_image = np.clip(filtered_image, 0, 1)
            filtered_image = img_as_ubyte(filtered_image)
            
            mse = mean_squared_error(img_cam, filtered_image)
            psnr = peak_signal_noise_ratio(img_cam, filtered_image)
            ssim = structural_similarity(img_cam, filtered_image)
            noisy_images[f'{filter_type.__name__} filter\nMSE: {mse:.1f} PSNR: {psnr:.1f} SSIM: {ssim:.3f}'] = filtered_image
        display_images(noisy_images, 5)

### 4. Методы выделения контуров на изображении

Контуры извлекаю несколькими классическими операторами: Робертса, Превитта, Собеля, Лапласа и детектором Кэнни. Последний обеспечивает наиболее аккуратные границы на выбранном изображении, хотя и требует настройки параметров. Остальные фильтры дают более шумные результаты и чувствительны к выбору порога.



In [ ]:
img_deer_gray = img_as_ubyte(color.rgb2gray(img_deer))

contour_filters = [roberts, prewitt, sobel, laplace, canny]
contour_thresholds = [0.01, 0.05, 0.1, 0.15, 0.2]
canny_params = [1, 2, 3, 4, 5]

for contour_filter in contour_filters:
    contour_images = {}
    if contour_filter == canny:
        for canny_param in canny_params:
            contoured_image = contour_filter(img_deer_gray, canny_param)
            mse = mean_squared_error(img_deer_edges, contoured_image)
            contour_images[f'{contour_filter.__name__} filter, sigma: {canny_param}\nMSE: {mse:.4f}'] = contoured_image
    else:
        contoured_image = contour_filter(img_deer_gray)
        for threshold in contour_thresholds:
            contoured_image_threshold = contoured_image > threshold
            mse = mean_squared_error(img_deer_edges, contoured_image_threshold)
            contour_images[f'{contour_filter.__name__} filter, threshold: {threshold}\nMSE: {mse:.4f}'] = contoured_image_threshold

    display_images(contour_images, len(contour_filters))

#### Исследование качества выделения контуров от уровня шума

Дополнительно изучаю, как наличие шумов ухудшает обнаружение границ. Для каждого типа и уровня шума сравниваю выход разных операторов. В большинстве случаев метод Кэнни сохраняет приемлемый баланс между полнотой и точностью, в то время как фильтры Робертса, Превитта и Собеля нужно тщательно подстраивать под уровень и характер помех. Лапласиан сильнее подчеркивает шум и требует предварительного сглаживания.



In [ ]:
img_deer_gray_noised = add_noise_to_image(img_deer_gray, 'gaussian', 0.001)

contour_filters = [roberts, prewitt, sobel, laplace, canny]
contour_thresholds = [0.01, 0.05, 0.1, 0.15, 0.2]
canny_params = [1, 2, 3, 4, 5]

for contour_filter in contour_filters:
    contour_images = {}
    if contour_filter == canny:
        for canny_param in canny_params:
            contoured_image = contour_filter(img_deer_gray_noised, canny_param)
            mse = mean_squared_error(img_deer_edges, contoured_image)
            contour_images[f'{contour_filter.__name__} filter, sigma: {canny_param}\nMSE: {mse:.4f}'] = contoured_image
    else:
        contoured_image = contour_filter(img_deer_gray_noised)
        for threshold in contour_thresholds:
            contoured_image_threshold = contoured_image > threshold
            mse = mean_squared_error(img_deer_edges, contoured_image_threshold)
            contour_images[f'{contour_filter.__name__} filter, threshold: {threshold}\nMSE: {mse:.4f}'] = contoured_image_threshold

    display_images(contour_images, len(contour_filters))

#### Шум типа "Соль"

Импульсы высокой яркости заполняют фон отдельными белыми точками. Медианный фильтр заметно сглаживает такие всплески и позволяет операторам Собеля и Превитта восстанавливать основные контуры, однако порог всё же приходится снижать, чтобы не потерять тонкие линии.



In [ ]:
img_deer_gray_noised = add_noise_to_image(img_deer_gray, 'salt', 0.03)

contour_filters = [roberts, prewitt, sobel, laplace, canny]
contour_thresholds = [0.01, 0.05, 0.1, 0.15, 0.2]
canny_params = [1, 2, 3, 4, 5]

for contour_filter in contour_filters:
    contour_images = {}
    if contour_filter == canny:
        for canny_param in canny_params:
            contoured_image = contour_filter(img_deer_gray_noised, canny_param)
            mse = mean_squared_error(img_deer_edges, contoured_image)
            contour_images[f'{contour_filter.__name__} filter, sigma: {canny_param}\nMSE: {mse:.4f}'] = contoured_image
    else:
        contoured_image = contour_filter(img_deer_gray_noised)
        for threshold in contour_thresholds:
            contoured_image_threshold = contoured_image > threshold
            mse = mean_squared_error(img_deer_edges, contoured_image_threshold)
            contour_images[f'{contour_filter.__name__} filter, threshold: {threshold}\nMSE: {mse:.4f}'] = contoured_image_threshold

    display_images(contour_images, len(contour_filters))

#### Шум типа "Перец"

Темные импульсы ведут себя аналогично, но делают изображение визуально темнее. При корректной предварительной фильтрации детектор Кэнни остаётся стабильным, а градиентные фильтры сохраняют границы лишь при умеренных пороговых значениях.



In [ ]:
img_deer_gray_noised = add_noise_to_image(img_deer_gray, 'pepper', 0.03)

contour_filters = [roberts, prewitt, sobel, laplace, canny]
contour_thresholds = [0.01, 0.05, 0.1, 0.15, 0.2]
canny_params = [1, 2, 3, 4, 5]

for contour_filter in contour_filters:
    contour_images = {}
    if contour_filter == canny:
        for canny_param in canny_params:
            contoured_image = contour_filter(img_deer_gray_noised, canny_param)
            mse = mean_squared_error(img_deer_edges, contoured_image)
            contour_images[f'{contour_filter.__name__} filter, sigma: {canny_param}\nMSE: {mse:.4f}'] = contoured_image
    else:
        contoured_image = contour_filter(img_deer_gray_noised)
        for threshold in contour_thresholds:
            contoured_image_threshold = contoured_image > threshold
            mse = mean_squared_error(img_deer_edges, contoured_image_threshold)
            contour_images[f'{contour_filter.__name__} filter, threshold: {threshold}\nMSE: {mse:.4f}'] = contoured_image_threshold

    display_images(contour_images, len(contour_filters))

#### Случайный импульсный шум

Когда шум заполняет кадр и светлыми, и тёмными выбросами, медианный фильтр справляется хуже, и на контурных изображениях появляются пробелы. Тем не менее Кэнни с повышенным значением сигмы всё ещё позволяет выделить крупные границы, тогда как операторы первого порядка теряют структуру объекта.



In [ ]:
img_deer_gray_noised = add_noise_to_image(img_deer_gray, 'impulse', 0.98)

contour_filters = [roberts, prewitt, sobel, laplace, canny]
contour_thresholds = [0.01, 0.05, 0.1, 0.15, 0.2]
canny_params = [1, 2, 3, 4, 5]

for contour_filter in contour_filters:
    contour_images = {}
    if contour_filter == canny:
        for canny_param in canny_params:
            contoured_image = contour_filter(img_deer_gray_noised, canny_param)
            mse = mean_squared_error(img_deer_edges, contoured_image)
            contour_images[f'{contour_filter.__name__} filter, sigma: {canny_param}\nMSE: {mse:.4f}'] = contoured_image
    else:
        contoured_image = contour_filter(img_deer_gray_noised)
        for threshold in contour_thresholds:
            contoured_image_threshold = contoured_image > threshold
            mse = mean_squared_error(img_deer_edges, contoured_image_threshold)
            contour_images[f'{contour_filter.__name__} filter, threshold: {threshold}\nMSE: {mse:.4f}'] = contoured_image_threshold

    display_images(contour_images, len(contour_filters))

### 5. Методы бинаризации изображений

На примере текстовой страницы сравниваю глобальные и локальные стратегии пороговой обработки. Цель — получить читаемое бинарное изображение при разнообразном фоне и неравномерном освещении.



In [ ]:
fig, ax = try_all_threshold(img_page, figsize=(8, 10), verbose=False)

plt.show()

#### Пороговая бинаризация

Перебираю линейку фиксированных порогов и подбираю значение, которое удерживает буквы и не заливает фон. Результат сильно зависит от выбранной границы — при слишком низком пороге фон темнеет, при высоком часть символов пропадает.



In [ ]:
thresholds = np.linspace(20, np.max(img_page), 8, endpoint=False)
binarized_images_thresholds = {}
for threshold in thresholds:
    binarized_page = img_as_ubyte(img_page >= threshold)
    binarized_images_thresholds[f'Threshold: {threshold:.0f}'] = (binarized_page, img_page, threshold)
display_images_with_thresholds(binarized_images_thresholds, 4)

#### Метод Оцу

Автоматический метод Оцу вычисляет порог, минимизирующий внутриклассовую дисперсию. Для данного образца он отдаёт предпочтение фону и «съедает» тонкие элементы текста, поэтому дополнительно проверяю локальные методы.



In [ ]:
threshold = threshold_otsu(img_page)
binarized_image = img_as_ubyte(img_page >= threshold)
binarized_images_thresholds = {f'Threshold Otsu: {threshold:.0f}': (binarized_image, img_page, threshold)}
display_images_with_thresholds(binarized_images_thresholds, 1)

#### Локальные методы бинаризации

Локальные стратегии — Ниблэк, Саувола, адаптивный порог и локальный Оцу — подстраивают порог под освещённость каждой области. Это особенно заметно в затемнённой левой части страницы: локальные методы удерживают структуру символов и улучшают читаемость по сравнению с глобальными подходами.



In [ ]:
block_size = 35

thresh_local = threshold_local(img_page, block_size=block_size, offset=10)
thresh_otsu_local = rank.otsu(img_page, disk(block_size))
thresh_niblack = threshold_niblack(img_page, window_size=block_size, k=0.8)
thresh_sauvola_local = threshold_sauvola(img_page, window_size=block_size)

binarized_images_local_thresholds = dict()
binarized_images_local_thresholds['Original'] = img_page
binarized_images_local_thresholds['Threshold Local'] = img_as_ubyte(img_page >= thresh_local)
binarized_images_local_thresholds['Threshold Otsu Local'] = img_as_ubyte(img_page >= thresh_otsu_local)
binarized_images_local_thresholds['Threshold Niblack'] = img_as_ubyte(img_page >= thresh_niblack)
binarized_images_local_thresholds['Threshold Sauvola'] = img_as_ubyte(img_page >= thresh_sauvola_local)

display_images(binarized_images_local_thresholds, 5)

## Вывод

Перепробованы основные операции обработки изображений: от коррекции цвета и яркости до фильтрации шумов, поиска контуров и бинаризации. Практика показала, что подбор метода сильно зависит от характера данных: медианная фильтрация и детектор Кэнни устойчивее к шумам, а адаптивные пороговые алгоритмы лучше справляются с неравномерным освещением.

